# 02 — EDA & Coverage Analysis
**Project:** Dark Store Placement Intelligence
**Author:** Ishaan Mankotia

In [ ]:
import sys
sys.path.insert(0, '../src')
from data_loader import fetch_dark_stores, generate_demand_grid
from opportunity_model import compute_nearest_store_distance, flag_covered_zones
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

In [ ]:
stores = fetch_dark_stores()
zones  = generate_demand_grid()
zones  = compute_nearest_store_distance(zones, stores)
zones  = flag_covered_zones(zones, coverage_radius_km=1.5)

## 2.1 Coverage Gap by Zone Type

In [ ]:
ct = zones.groupby(['zone_type','store_covered']).size().unstack(fill_value=0)
ct.columns = ['Uncovered','Covered']
ct.plot.bar(color=['#E74C3C','#2ECC71'], edgecolor='white', figsize=(9,5))
plt.title('Coverage by Zone Type'); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 2.2 Distance Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.histplot(zones['nearest_store_dist_km'], bins=30, ax=axes[0], color='steelblue')
axes[0].axvline(1.5, color='red', linestyle='--', label='1.5km threshold')
axes[0].legend(); axes[0].set_title('Distance to Nearest Dark Store')
zones['store_covered'].value_counts().rename({True:'Covered',False:'Uncovered'}).plot.pie(
    ax=axes[1], autopct='%1.0f%%', colors=['#2ECC71','#E74C3C'], startangle=90)
axes[1].set_title('Coverage Split'); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

## 2.3 Income vs Density

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
scatter = ax.scatter(zones['income_proxy'], zones['population_density']/1000,
                     c=zones['nearest_store_dist_km'], cmap='RdYlGn_r', alpha=0.6, s=30)
plt.colorbar(scatter, ax=ax, label='Distance to nearest store (km)')
ax.set_xlabel('Income Proxy (0–1)'); ax.set_ylabel('Population Density (thousands/km²)')
ax.set_title('Income vs Density — coloured by coverage gap')
plt.tight_layout(); plt.show()